# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [ ]:
import os
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
# session time zone = UTC so that timestamps are displayed exactly as stored
# in the parquet files (NYC local time) and not shifted by the machine TZ

sc = spark.sparkContext
print("Spark version:", spark.version)

In [ ]:
def download(url, path):
    """Download url to path (skipped if the file already exists)."""
    if os.path.exists(path):
        print(f"{path} already downloaded")
        return path
    response = requests.get(url, timeout=300)
    # check that response was good and save the data
    if response.status_code != 200:
        raise RuntimeError(f"download failed ({response.status_code}): {url}")
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"{path} downloaded ({len(response.content) / 1e6:.1f} MB)")
    return path


BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

# set dl url for January 2019 trip data
download_url = f"{BASE_URL}/yellow_tripdata_2019-01.parquet"
jan_2019_trip_data = download(download_url, "yellow_tripdata_2019-01.parquet")

In [ ]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [ ]:
# Show the dataframe
df_trips.show(5)

In [ ]:
df_trips.printSchema()
print("rows:", df_trips.count())

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

> **Language choice:** the main lab is done with the **PySpark DataFrame API**.
> Part 3 re-answers some questions in **pure Spark SQL**.

#### Preparation – unique key and derived columns

In [ ]:
def enrich(df):
    """Add a unique trip key and the derived columns used in the lab."""
    pickup, dropoff = "tpep_pickup_datetime", "tpep_dropoff_datetime"
    return (df
        # unique key: monotonically_increasing_id is guaranteed unique
        # (not consecutive) and needs no shuffle, unlike row_number()
        .withColumn("trip_id", F.monotonically_increasing_id())
        # same types whatever the year of the file
        .withColumn("passenger_count", F.col("passenger_count").cast("double"))
        .withColumn("PULocationID", F.col("PULocationID").cast("int"))
        .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
        .withColumn("pickup_date", F.to_date(pickup))
        .withColumn("pickup_hour", F.hour(pickup))
        .withColumn("day_of_week", F.date_format(pickup, "EEEE"))
        # 1 = Sunday ... 7 = Saturday (used to sort the days)
        .withColumn("dow_num", F.dayofweek(pickup))
        .withColumn("duration_min",
                    (F.unix_timestamp(dropoff) - F.unix_timestamp(pickup)) / 60)
        .withColumn("time_of_day",
            F.when(F.col("pickup_hour") < 6, "1-late night (0-5h)")
             .when(F.col("pickup_hour") < 12, "2-morning (6-11h)")
             .when(F.col("pickup_hour") < 17, "3-afternoon (12-16h)")
             .when(F.col("pickup_hour") < 22, "4-evening (17-21h)")
             .otherwise("5-night (22-23h)")))


def in_month(df, year, month):
    """Keep only the trips picked up during the given month."""
    return df.filter((F.year("tpep_pickup_datetime") == year) &
                     (F.month("tpep_pickup_datetime") == month))


def clean(df):
    """Keep only plausible trips (see the outlier analysis at the end)."""
    return df.filter(
        (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) &
        (F.col("fare_amount") > 0) & (F.col("fare_amount") < 500) &
        (F.col("duration_min") >= 1) & (F.col("duration_min") <= 180) &
        # passenger_count can be null (not reported): kept
        (F.col("passenger_count").isNull() |
         F.col("passenger_count").between(1, 6)) &
        # average speed below 80 mph
        (F.col("trip_distance") / (F.col("duration_min") / 60) < 80))


trips = enrich(df_trips).cache()
trips_jan = in_month(trips, 2019, 1).cache()
trips_clean = clean(trips_jan).cache()

print(f"all rows            : {trips.count():>10,}")
print(f"picked up in 01/2019: {trips_jan.count():>10,}")
print(f"plausible trips     : {trips_clean.count():>10,}")
trips.select("trip_id", "tpep_pickup_datetime", "pickup_date", "pickup_hour",
             "day_of_week", "duration_min", "time_of_day").show(5, truncate=False)

In [ ]:
# the key is unique: as many distinct ids as rows
trips.select(F.countDistinct("trip_id").alias("distinct_ids"),
             F.count("*").alias("rows")).show()

#### Which trip has the highest passenger count

In [ ]:
max_pass = trips.agg(F.max("passenger_count")).first()[0]
print("max passenger count:", max_pass)
print("number of trips with it:",
      trips.filter(F.col("passenger_count") == max_pass).count())

trips.orderBy(F.desc("passenger_count"), F.desc("trip_distance")) \
     .select("trip_id", "tpep_pickup_datetime", "passenger_count",
             "trip_distance", "fare_amount", "total_amount") \
     .show(10)

In [ ]:
# distribution of passenger_count
trips.groupBy("passenger_count").count().orderBy("passenger_count").show()

#### Average passenger count

In [ ]:
trips.agg(
    F.round(F.avg("passenger_count"), 3).alias("avg_all_trips"),
    F.round(F.avg(F.when(F.col("passenger_count") > 0,
                         F.col("passenger_count"))), 3)
     .alias("avg_excluding_0"),
    F.round(F.avg(F.when(F.col("passenger_count").between(1, 6),
                         F.col("passenger_count"))), 3)
     .alias("avg_1_to_6_only"),
).show()

The average is about 1.5 passengers per trip: most taxi rides carry a single
person. `avg()` ignores nulls; trips with 0 passengers (recording errors) slightly
lower the average, so the version restricted to 1–6 passengers is the most reliable.

#### Shortest / longest trip by distance and by time

In [ ]:
cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "trip_distance", "duration_min", "fare_amount", "total_amount"]

print("Longest trips by distance (raw data)")
trips.orderBy(F.desc("trip_distance")).select(cols).show(5)

print("Shortest trips by distance (raw data)")
trips.orderBy("trip_distance").select(cols).show(5)
print("trips with distance = 0:",
      trips.filter(F.col("trip_distance") == 0).count())

print("Shortest real trip (distance > 0)")
trips_clean.orderBy("trip_distance", "duration_min").select(cols).show(5)

In [ ]:
print("Longest trips by time (raw data)")
trips.orderBy(F.desc("duration_min")).select(cols).show(5)

print("Shortest trips by time (raw data)")
trips.orderBy("duration_min").select(cols).show(5)
print("trips with duration <= 0:",
      trips.filter(F.col("duration_min") <= 0).count())

print("Longest / shortest plausible trips (cleaned data)")
trips_clean.orderBy(F.desc("duration_min")).select(cols).show(3)
trips_clean.orderBy("duration_min").select(cols).show(3)

In the raw data the extremes are **not real trips**: distances of several thousand
miles, trips of 0 miles or 0 seconds, drop-off before pick-up (negative duration) or
trips lasting several days (meter left running). The meaningful answers are the ones on
the cleaned data (details in the outlier section).

#### Busiest / slowest single day

In [ ]:
daily = trips_jan.groupBy("pickup_date", "day_of_week").count() \
                .withColumnRenamed("count", "trips")

print("Busiest days")
daily.orderBy(F.desc("trips")).show(5)
print("Slowest days")
daily.orderBy("trips").show(5)

Only trips picked up in January 2019 are used (the file contains a few trips dated
2008, 2018, 2088...). The slowest days are usually public holidays / weekends such as
**Jan 1st** (New Year) and Sundays; the busiest are weekdays late in the week.

#### Busiest / slowest time of day

In [ ]:
hourly = trips_jan.groupBy("pickup_hour").count() \
                 .withColumnRenamed("count", "trips").orderBy("pickup_hour")
hourly.show(24)

print("busiest hour:", hourly.orderBy(F.desc("trips")).first())
print("slowest hour:", hourly.orderBy("trips").first())

In [ ]:
# buckets do not all have the same length -> compare trips per hour too
n_hours = {"1-late night (0-5h)": 6, "2-morning (6-11h)": 6,
           "3-afternoon (12-16h)": 5, "4-evening (17-21h)": 5,
           "5-night (22-23h)": 2}
hours_map = F.create_map(*[x for k, v in n_hours.items()
                           for x in (F.lit(k), F.lit(v))])

trips_jan.groupBy("time_of_day").count() \
    .withColumn("trips_per_hour",
                F.round(F.col("count") / hours_map[F.col("time_of_day")])) \
    .orderBy("time_of_day").show(truncate=False)

The evening rush (around 18–19h) is the busiest period and 4–5 am the slowest.
Because the buckets have different lengths, the *trips per hour* column is the fair way
to compare them.

#### On average, which day of the week is slowest / busiest

In [ ]:
# January 2019 has 5 Tuesdays, Wednesdays and Thursdays but only 4 of the other
# days -> a plain count per weekday would be biased: average the daily counts
dow_avg = daily.groupBy("day_of_week") \
    .agg(F.count("*").alias("nb_days"),
         F.round(F.avg("trips")).alias("avg_trips_per_day")) \
    .orderBy(F.desc("avg_trips_per_day"))
dow_avg.show()

#### Does trip distance or number of passengers affect the tip amount

In [ ]:
# tips are only recorded for credit-card payments (payment_type = 1);
# cash tips are always 0 in the data -> analyse card payments only
card = trips_clean.filter(F.col("payment_type") == 1)

print("corr(distance, tip)   :", round(card.stat.corr("trip_distance", "tip_amount"), 3))
print("corr(passengers, tip) :",
      round(card.na.drop(subset=["passenger_count"])
                .stat.corr("passenger_count", "tip_amount"), 3))
print("corr(fare, tip)       :", round(card.stat.corr("fare_amount", "tip_amount"), 3))

In [ ]:
dist_bucket = (F.when(F.col("trip_distance") < 1, "a. < 1 mi")
               .when(F.col("trip_distance") < 3, "b. 1-3 mi")
               .when(F.col("trip_distance") < 5, "c. 3-5 mi")
               .when(F.col("trip_distance") < 10, "d. 5-10 mi")
               .when(F.col("trip_distance") < 20, "e. 10-20 mi")
               .otherwise("f. 20+ mi"))

tip_by_distance = card.withColumn("distance_bucket", dist_bucket) \
    .groupBy("distance_bucket") \
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
         F.round(F.avg(F.col("tip_amount") / F.col("fare_amount") * 100), 1)
          .alias("avg_tip_pct_of_fare")) \
    .orderBy("distance_bucket")
tip_by_distance.show()

tip_by_passengers = card.groupBy("passenger_count") \
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
         F.round(F.avg(F.col("tip_amount") / F.col("fare_amount") * 100), 1)
          .alias("avg_tip_pct_of_fare")) \
    .orderBy("passenger_count")
tip_by_passengers.show()

- **Distance → yes.** The correlation between distance and tip is strongly positive:
  the tip is mostly a percentage of the fare, and the fare grows with distance. The tip
  *as a percentage of the fare* stays roughly stable (around 15–20 %), which shows that
  customers tip proportionally, not per mile.
- **Passengers → practically no.** The correlation is close to 0 and the average tip is
  almost the same whatever the number of passengers: a group does not tip more than a
  single passenger for the same ride.
- Only card payments are used, because cash tips are not recorded (always 0), which
  would bias the averages downward.

#### Highest "extra" charge and which trip

In [ ]:
trips.orderBy(F.desc("extra")) \
     .select("trip_id", "tpep_pickup_datetime", "trip_distance", "fare_amount",
             "extra", "total_amount", "PULocationID", "DOLocationID") \
     .show(5)

# normal values of extra (rush hour / overnight surcharges of $0.50 / $1)
trips.groupBy("extra").count().orderBy(F.desc("count")).show(10)

In 2019 the `extra` field should only contain the $0.50 overnight and $1 rush-hour
surcharges (and $2.50 congestion surcharge from February 2019). The highest values found
(tens or hundreds of dollars) are therefore data-entry errors and are clearly outliers.

#### Strange data points / outliers

In [ ]:
trips.select("passenger_count", "trip_distance", "duration_min", "fare_amount",
             "extra", "tip_amount", "tolls_amount", "total_amount") \
     .summary("min", "1%", "50%", "99%", "max").show()

In [ ]:
checks = {
    "pickup not in Jan 2019":
        ~((F.year("tpep_pickup_datetime") == 2019) &
          (F.month("tpep_pickup_datetime") == 1)),
    "passenger_count = 0": F.col("passenger_count") == 0,
    "passenger_count > 6": F.col("passenger_count") > 6,
    "trip_distance = 0": F.col("trip_distance") == 0,
    "trip_distance > 100 mi": F.col("trip_distance") > 100,
    "duration <= 0": F.col("duration_min") <= 0,
    "duration > 3 h": F.col("duration_min") > 180,
    "speed > 80 mph": (F.col("duration_min") > 0) &
        (F.col("trip_distance") / (F.col("duration_min") / 60) > 80),
    "fare_amount < 0": F.col("fare_amount") < 0,
    "fare_amount > 500": F.col("fare_amount") > 500,
    "extra < 0 or > 5": (F.col("extra") < 0) | (F.col("extra") > 5),
    "tip > fare": F.col("tip_amount") > F.col("fare_amount") * 1.0,
}

outliers = trips.agg(*[F.sum(cond.cast("int")).alias(name)
                       for name, cond in checks.items()])
total = trips.count()
rows = [(name, int(v or 0), round(100 * (v or 0) / total, 3))
        for name, v in outliers.first().asDict().items()]
spark.createDataFrame(rows, ["check", "trips", "pct"]).show(truncate=False)

In [ ]:
# examples: dates outside January 2019
trips.filter(~((F.year("tpep_pickup_datetime") == 2019) &
               (F.month("tpep_pickup_datetime") == 1))) \
     .groupBy(F.year("tpep_pickup_datetime").alias("year"),
              F.month("tpep_pickup_datetime").alias("month")) \
     .count().orderBy("year", "month").show()

**Outliers – reasoning**

| Anomaly | Why it is strange |
|---|---|
| Pick-up dates outside January 2019 (2008, 2009, 2018, 2088...) | The file is supposed to contain January 2019 only: wrong clock on the taximeter. Trips from the last days of Dec 2018 are "late" records. |
| `passenger_count` = 0 or > 6 | A yellow cab cannot legally carry more than 5–6 passengers; 0 means the driver did not enter the value. |
| `trip_distance` = 0 | Cancelled trips, meter started by mistake, or GPS problem – yet a fare is often charged. |
| Distances > 100 miles / speeds > 80 mph | Physically impossible in NYC traffic in the recorded time → odometer/GPS error. |
| Duration ≤ 0 or of several days | Drop-off before pick-up, or meter never switched off. |
| Negative amounts (`fare_amount`, `extra`, `total_amount` < 0) | Refunds, voided or disputed transactions (payment types 3/4). They are not trips. |
| Huge `extra` / fares, tip > fare | Typing errors of the driver, or very rare genuine cases (large tips); they distort averages. |

These points represent a small share of the data (see the percentages above) but they
are enough to distort min / max / averages. That is why the `clean()` function is used
for the averages, and why raw and cleaned results are both shown for min/max questions.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [ ]:
# load the taxi zone lookup (small csv -> explicit schema, as recommended)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

zone_path = download("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
                     "taxi_zone_lookup.csv")

zone_schema = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .schema(zone_schema) \
    .load(zone_path)

df_zones.show(5)
df_zones.groupBy("Borough").count().orderBy(F.desc("count")).show()

In [ ]:
def add_boroughs(df):
    """Join pick-up and drop-off boroughs/zones (zones table is tiny ->
    broadcast join, no shuffle of the big trips table)."""
    pu = df_zones.select(F.col("LocationID").alias("PULocationID"),
                         F.col("Borough").alias("pu_borough"),
                         F.col("Zone").alias("pu_zone"))
    do = df_zones.select(F.col("LocationID").alias("DOLocationID"),
                         F.col("Borough").alias("do_borough"),
                         F.col("Zone").alias("do_zone"))
    return (df.join(F.broadcast(pu), "PULocationID", "left")
              .join(F.broadcast(do), "DOLocationID", "left"))


trips_b = add_boroughs(trips_jan).cache()          # all Jan 2019 trips
trips_b_clean = add_boroughs(trips_clean).cache()  # plausible trips only
trips_b.select("trip_id", "pu_borough", "pu_zone", "do_borough", "do_zone").show(5)

#### Which borough had the most pickups? dropoffs?

In [ ]:
total_jan = trips_b.count()

pickups = trips_b.groupBy(F.col("pu_borough").alias("borough")) \
    .agg(F.count("*").alias("pickups"))
dropoffs = trips_b.groupBy(F.col("do_borough").alias("borough")) \
    .agg(F.count("*").alias("dropoffs"))

borough_counts = pickups.join(dropoffs, "borough", "outer") \
    .withColumn("pickups_pct", F.round(100 * F.col("pickups") / total_jan, 2)) \
    .withColumn("dropoffs_pct", F.round(100 * F.col("dropoffs") / total_jan, 2)) \
    .orderBy(F.desc("pickups"))
borough_counts.show()

**Manhattan** dominates both pickups and dropoffs by far (around 90 % of pickups):
yellow cabs mostly serve Manhattan, while the outer boroughs are served by green cabs
and ride-hailing apps. Queens comes second thanks to the airports (JFK, LaGuardia).
Dropoffs are a bit more spread out than pickups (people take a cab from Manhattan to go
home in Brooklyn/Queens). "Unknown" / "N/A" are the location ids 264/265 (unknown zone).

#### Busy / slow times by borough

In [ ]:
by_b_hour = trips_b.groupBy("pu_borough", "pickup_hour").count() \
                   .withColumnRenamed("count", "trips")

w_desc = Window.partitionBy("pu_borough").orderBy(F.desc("trips"))
w_asc = Window.partitionBy("pu_borough").orderBy("trips")

busy_slow = by_b_hour \
    .withColumn("rank_busy", F.row_number().over(w_desc)) \
    .withColumn("rank_slow", F.row_number().over(w_asc))

busiest = busy_slow.filter("rank_busy = 1") \
    .select("pu_borough", F.col("pickup_hour").alias("busiest_hour"),
            F.col("trips").alias("trips_busiest"))
slowest = busy_slow.filter("rank_slow = 1") \
    .select("pu_borough", F.col("pickup_hour").alias("slowest_hour"),
            F.col("trips").alias("trips_slowest"))

busiest.join(slowest, "pu_borough").orderBy("pu_borough").show()

In [ ]:
# same question with the time-of-day buckets (share of the borough's trips)
tod = trips_b.groupBy("pu_borough").pivot("time_of_day").count().na.fill(0)
tod_cols = [c for c in tod.columns if c != "pu_borough"]
row_total = sum(F.col(f"`{c}`") for c in tod_cols)
tod.select("pu_borough",
           *[F.round(100 * F.col(f"`{c}`") / row_total, 1).alias(c)
             for c in tod_cols]) \
   .orderBy("pu_borough").show(truncate=False)

In Manhattan the peak is in the evening rush (18–19h). For Queens the airports
change the profile: activity is more spread through the day, with peaks linked to
flight arrivals (afternoon / evening) and relatively more late-night trips. Everywhere
the slowest time is the early morning (≈ 4–5 am).

#### Busiest days of the week by borough

In [ ]:
# average number of pickups per day, per borough and day of week
b_daily = trips_b.groupBy("pu_borough", "pickup_date", "day_of_week", "dow_num") \
                 .count()
b_dow = b_daily.groupBy("pu_borough", "day_of_week", "dow_num") \
    .agg(F.round(F.avg("count")).alias("avg_trips_per_day"))

w = Window.partitionBy("pu_borough").orderBy(F.desc("avg_trips_per_day"))
b_dow.withColumn("rank", F.row_number().over(w)) \
     .filter("rank = 1 or rank = 7") \
     .withColumn("type", F.when(F.col("rank") == 1, "busiest").otherwise("slowest")) \
     .select("pu_borough", "type", "day_of_week", "avg_trips_per_day") \
     .orderBy("pu_borough", "type").show(20)

# full table (days in columns)
b_dow.groupBy("pu_borough").pivot("dow_num", list(range(1, 8))) \
     .agg(F.first("avg_trips_per_day")) \
     .toDF("pu_borough", "Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat") \
     .orderBy("pu_borough").show()

#### Average trip distance and average fare by borough

In [ ]:
# averages computed on plausible trips only (outliers removed)
borough_avgs = trips_b_clean.groupBy("pu_borough") \
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("total_amount"), 2).alias("avg_total"),
         F.round(F.avg("duration_min"), 1).alias("avg_duration_min")) \
    .orderBy(F.desc("trips"))
borough_avgs.show()

Manhattan trips are the shortest and cheapest on average (dense area, short hops),
whereas trips starting in Queens are much longer and more expensive because many of
them are airport trips (JFK → Manhattan has a flat fare of $52 in 2019). EWR (Newark
airport) has very few pickups but high fares.

#### Highest / lowest fare amounts and their boroughs

In [ ]:
fare_cols = ["trip_id", "tpep_pickup_datetime", "fare_amount", "total_amount",
             "trip_distance", "pu_borough", "do_borough"]

print("Highest fares (raw)")
trips_b.orderBy(F.desc("fare_amount")).select(fare_cols).show(5)
print("Lowest fares (raw) - negative = refunds / voided trips")
trips_b.orderBy("fare_amount").select(fare_cols).show(5)

print("Highest / lowest fares among plausible trips")
trips_b_clean.orderBy(F.desc("fare_amount")).select(fare_cols).show(5)
trips_b_clean.orderBy("fare_amount").select(fare_cols).show(5)

#### Most recent January – has any average metric changed?

In [ ]:
# most recently available January: TLC publishes files with ~2 months of
# delay, so we try the latest years first
def latest_january(candidates=(2026, 2025, 2024)):
    for year in candidates:
        path = f"yellow_tripdata_{year}-01.parquet"
        if os.path.exists(path):
            return year
        try:
            r = requests.head(f"{BASE_URL}/{path}", timeout=30)
            if r.status_code == 200:
                return year
        except requests.RequestException:
            pass
    raise RuntimeError("no recent January file found")


RECENT_YEAR = latest_january()
recent_path = download(f"{BASE_URL}/yellow_tripdata_{RECENT_YEAR}-01.parquet",
                       f"yellow_tripdata_{RECENT_YEAR}-01.parquet")

df_recent = spark.read.parquet(recent_path)
df_recent.printSchema()

trips_recent = in_month(enrich(df_recent), RECENT_YEAR, 1).cache()
trips_recent_clean = clean(trips_recent).cache()
trips_recent_b_clean = add_boroughs(trips_recent_clean).cache()
print(f"Jan {RECENT_YEAR}: {trips_recent.count():,} trips "
      f"({trips_recent_clean.count():,} plausible)")

In [ ]:
def global_metrics(df_all, df_clean, label):
    n_days = df_all.select(F.countDistinct("pickup_date")).first()[0]
    card_ = df_clean.filter(F.col("payment_type") == 1)
    m = df_clean.agg(
        F.avg("passenger_count").alias("avg_passengers"),
        F.avg("trip_distance").alias("avg_distance_mi"),
        F.avg("duration_min").alias("avg_duration_min"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("extra").alias("avg_extra"),
        F.avg("total_amount").alias("avg_total"),
    ).first().asDict()
    m["avg_tip_card"] = card_.agg(F.avg("tip_amount")).first()[0]
    m["pct_card_payments"] = 100 * card_.count() / df_clean.count()
    m["trips_per_day"] = df_all.count() / n_days
    return {k: round(v, 2) for k, v in m.items()}


import pandas as pd

m_2019 = global_metrics(trips_jan, trips_clean, "2019")
m_recent = global_metrics(trips_recent, trips_recent_clean, str(RECENT_YEAR))
comparison = pd.DataFrame({"Jan 2019": m_2019, f"Jan {RECENT_YEAR}": m_recent})
comparison["change %"] = (100 * (comparison[f"Jan {RECENT_YEAR}"] /
                                 comparison["Jan 2019"] - 1)).round(1)
comparison

In [ ]:
# by borough (pick-up borough)
recent_avgs = trips_recent_b_clean.groupBy("pu_borough") \
    .agg(F.count("*").alias("trips_recent"),
         F.round(F.avg("trip_distance"), 2).alias("avg_distance_recent"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare_recent"))

borough_avgs.select("pu_borough", F.col("trips").alias("trips_2019"),
                    F.col("avg_distance_mi").alias("avg_distance_2019"),
                    F.col("avg_fare").alias("avg_fare_2019")) \
    .join(recent_avgs, "pu_borough", "outer") \
    .withColumn("fare_change_pct",
                F.round(100 * (F.col("avg_fare_recent") /
                               F.col("avg_fare_2019") - 1), 1)) \
    .orderBy(F.desc("trips_2019")).show()

**What changed (compare with the tables above):**

- **Far fewer yellow-cab trips.** The number of trips per day is roughly halved
  compared to 2019: competition from Uber/Lyft and the post-Covid change in habits.
- **Higher fares.** The average fare and total amount increased strongly: the TLC fare
  increase of December 2022 (higher base fare and per-mile rate), the $2.50 congestion
  surcharge introduced in February 2019, and from January 5th 2025 the congestion
  pricing fee for trips in the Manhattan CBD (`cbd_congestion_fee` column, new in
  the schema).
- **Longer trips on average**: the remaining yellow-cab demand is relatively more
  oriented toward airports and longer rides.
- **More card payments**, and a higher average tip in dollars (tips follow the fare).
- The schema itself evolved (`Airport_fee`, `cbd_congestion_fee`, integer types for
  ids), which is why `enrich()` casts the columns before comparing.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [ ]:
# register the dataframes as temporary views for Spark SQL
trips_jan.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")
trips_recent.createOrReplaceTempView("trips_recent")

**SQL 1 – Which trip has the highest passenger count**

In [ ]:
spark.sql("""
    SELECT trip_id, tpep_pickup_datetime, passenger_count,
           trip_distance, total_amount
    FROM trips
    WHERE passenger_count = (SELECT MAX(passenger_count) FROM trips)
    ORDER BY trip_distance DESC
    LIMIT 10
""").show()

**SQL 2 – On average, which day of the week is busiest / slowest**

In [ ]:
spark.sql("""
    WITH daily AS (
        SELECT to_date(tpep_pickup_datetime)             AS pickup_date,
               date_format(tpep_pickup_datetime, 'EEEE') AS day_of_week,
               COUNT(*)                                  AS trips
        FROM trips
        GROUP BY 1, 2
    )
    SELECT day_of_week,
           COUNT(*)            AS nb_days,
           ROUND(AVG(trips))   AS avg_trips_per_day
    FROM daily
    GROUP BY day_of_week
    ORDER BY avg_trips_per_day DESC
""").show()

**SQL 3 (join) – Which borough had the most pickups / dropoffs**

In [ ]:
spark.sql("""
    WITH pu AS (
        SELECT z.Borough AS borough, COUNT(*) AS pickups
        FROM trips t
        LEFT JOIN zones z ON t.PULocationID = z.LocationID
        GROUP BY z.Borough
    ),
    do AS (
        SELECT z.Borough AS borough, COUNT(*) AS dropoffs
        FROM trips t
        LEFT JOIN zones z ON t.DOLocationID = z.LocationID
        GROUP BY z.Borough
    )
    SELECT COALESCE(pu.borough, do.borough) AS borough,
           pu.pickups,
           ROUND(100 * pu.pickups / SUM(pu.pickups) OVER (), 2) AS pickups_pct,
           do.dropoffs
    FROM pu FULL OUTER JOIN do ON pu.borough = do.borough
    ORDER BY pu.pickups DESC
""").show()

**SQL 4 (join) – Average distance and fare by borough, 2019 vs most recent January**

In [ ]:
clean_filter = """
    trip_distance > 0 AND trip_distance < 100
    AND fare_amount > 0 AND fare_amount < 500
    AND duration_min BETWEEN 1 AND 180
    AND (passenger_count IS NULL OR passenger_count BETWEEN 1 AND 6)
    AND trip_distance / (duration_min / 60) < 80
"""

spark.sql(f"""
    WITH y2019 AS (
        SELECT z.Borough AS borough,
               ROUND(AVG(t.trip_distance), 2) AS avg_distance_2019,
               ROUND(AVG(t.fare_amount), 2)   AS avg_fare_2019
        FROM trips t
        JOIN zones z ON t.PULocationID = z.LocationID
        WHERE {clean_filter}
        GROUP BY z.Borough
    ),
    recent AS (
        SELECT z.Borough AS borough,
               ROUND(AVG(t.trip_distance), 2) AS avg_distance_recent,
               ROUND(AVG(t.fare_amount), 2)   AS avg_fare_recent
        FROM trips_recent t
        JOIN zones z ON t.PULocationID = z.LocationID
        WHERE {clean_filter}
        GROUP BY z.Borough
    )
    SELECT a.borough, avg_distance_2019, avg_distance_recent,
           avg_fare_2019, avg_fare_recent,
           ROUND(100 * (avg_fare_recent / avg_fare_2019 - 1), 1) AS fare_change_pct
    FROM y2019 a
    JOIN recent b ON a.borough = b.borough
    ORDER BY a.borough
""").show()

The SQL results are identical to the PySpark ones: both APIs are compiled by the
same Catalyst optimizer into the same physical plan (the broadcast join of the small
`zones` table is chosen automatically in SQL too). `spark.sql(...).explain()` can be
used to check it.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

### Busiest season of 2019 (full year)

In [ ]:
# load the 12 months of 2019 (~100 MB each). Set to False to skip.
RUN_FULL_YEAR = True

if RUN_FULL_YEAR:
    monthly = []
    for month in range(1, 13):
        path = download(f"{BASE_URL}/yellow_tripdata_2019-{month:02d}.parquet",
                        f"yellow_tripdata_2019-{month:02d}.parquet")
        # only the pick-up time is needed -> parquet reads a single column
        monthly.append(in_month(spark.read.parquet(path)
                                .select("tpep_pickup_datetime"), 2019, month))

    from functools import reduce
    year_2019 = reduce(lambda a, b: a.unionByName(b), monthly)

    season = (F.when(F.month("tpep_pickup_datetime").isin(12, 1, 2), "winter")
               .when(F.month("tpep_pickup_datetime").isin(3, 4, 5), "spring")
               .when(F.month("tpep_pickup_datetime").isin(6, 7, 8), "summer")
               .otherwise("fall"))

    seasons = year_2019.withColumn("season", season) \
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
        .groupBy("season") \
        .agg(F.count("*").alias("trips"),
             F.countDistinct("pickup_date").alias("days")) \
        .withColumn("trips_per_day", F.round(F.col("trips") / F.col("days"))) \
        .orderBy(F.desc("trips_per_day"))
    seasons.show()

Seasons do not all have the same number of days (winter 2019 = Jan, Feb, Dec = 90
days vs 92 for spring/summer), so *trips per day* is the fair comparison. Yellow-cab
activity is usually highest in **spring** (March–May) and lowest in **summer**, when
many New Yorkers leave the city and tourists use other modes of transport.

### Visualizations (Spark 4 native plotting)

Since Spark 4.0, `DataFrame.plot` works directly on a Spark DataFrame (Plotly backend,
`pip install plotly` if needed): Spark aggregates, then only the small result is sent to
the plotting library.

In [ ]:
# 1. Pickups per hour of day (January 2019)
fig = hourly.plot.line(x="pickup_hour", y="trips",
                       title="Pickups per hour of day - January 2019")
fig.show()

In [ ]:
# 2. Average tip vs distance (card payments)
fig = tip_by_distance.plot.bar(x="distance_bucket", y="avg_tip",
                               title="Average tip by trip distance")
fig.show()

fig = tip_by_distance.plot.bar(x="distance_bucket", y="avg_tip_pct_of_fare",
                               title="Average tip as % of the fare by distance")
fig.show()

In [ ]:
# 3. Pickups by borough (log scale: Manhattan dwarfs the others)
fig = borough_counts.filter(F.col("pickups").isNotNull()) \
    .plot.bar(x="borough", y="pickups", title="Pickups by borough - January 2019")
fig.update_yaxes(type="log")
fig.show()

In [ ]:
# 4. Average trips per day by day of week
dow_order = daily.groupBy("day_of_week", F.dayofweek("pickup_date").alias("n")) \
    .agg(F.round(F.avg("trips")).alias("avg_trips_per_day")).orderBy("n")
fig = dow_order.plot.bar(x="day_of_week", y="avg_trips_per_day",
                         title="Average trips per day of week - January 2019")
fig.show()

In [ ]:
# 5. 2019 vs most recent January - average fare by borough
fare_cmp = borough_avgs.select("pu_borough", F.col("avg_fare").alias("Jan 2019")) \
    .join(recent_avgs.select("pu_borough",
                             F.col("avg_fare_recent").alias(f"Jan {RECENT_YEAR}")),
          "pu_borough") \
    .orderBy("pu_borough")
fig = fare_cmp.plot.bar(x="pu_borough", y=["Jan 2019", f"Jan {RECENT_YEAR}"],
                        title="Average fare by pick-up borough")
fig.update_layout(barmode="group")
fig.show()

In [ ]:
# 6. Busiest season of 2019
if RUN_FULL_YEAR:
    fig = seasons.plot.bar(x="season", y="trips_per_day",
                           title="Average yellow-cab trips per day by season (2019)")
    fig.show()

### Another dataset: green taxis vs yellow taxis

Green ("boro") taxis were created to serve the outer boroughs, where yellow cabs rarely
pick up. We load the green taxi data for the same recent January and compare where the
two fleets pick up their passengers.

In [ ]:
green_path = download(f"{BASE_URL}/green_tripdata_{RECENT_YEAR}-01.parquet",
                      f"green_tripdata_{RECENT_YEAR}-01.parquet")
df_green = spark.read.parquet(green_path) \
    .withColumn("PULocationID", F.col("PULocationID").cast("int"))
df_green = df_green.filter(
    (F.year("lpep_pickup_datetime") == RECENT_YEAR) &
    (F.month("lpep_pickup_datetime") == 1))


def share_by_borough(df, name):
    total = df.count()
    return df.join(F.broadcast(df_zones.select(
                       F.col("LocationID").alias("PULocationID"),
                       F.col("Borough").alias("borough"))),
                   "PULocationID", "left") \
             .groupBy("borough") \
             .agg(F.count("*").alias(f"{name}_trips")) \
             .withColumn(f"{name}_pct",
                         F.round(100 * F.col(f"{name}_trips") / total, 1))


fleet = share_by_borough(trips_recent, "yellow") \
    .join(share_by_borough(df_green, "green"), "borough", "outer") \
    .orderBy(F.desc("yellow_trips"))
fleet.show()

fig = fleet.na.fill(0).plot.bar(x="borough", y=["yellow_pct", "green_pct"],
                                title=f"Share of pickups by borough - Jan {RECENT_YEAR}")
fig.update_layout(barmode="group")
fig.show()

Yellow cabs are almost entirely a Manhattan (and airport) service, while green cabs
have a much larger share of their pickups in Brooklyn, Queens and the Bronx (and upper
Manhattan, since they are not allowed to pick up street hails in the Manhattan core).
The green fleet is also much smaller than the yellow one in number of trips.

In [ ]:
# free the cache and stop the session at the end of the lab
spark.catalog.clearCache()
spark.stop()